# LONDON

In [2]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, DeepGraphInfomax

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, f1_score, mean_squared_error
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

In [3]:
flows = pd.read_csv("../data/ODWP01EW_OA.csv")  # the travel-to-work data, 2021 Census
lookup_lsoa = pd.read_csv("../data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv") #the LSOA lookup file

C:\Users\jzvf437\AppData\Local\Temp\ipykernel_11588\3074991850.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup_lsoa = pd.read_csv("../data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv") #the LSOA lookup file


In [4]:
flows['Output Areas code'] = flows['Output Areas code'].astype(str).str.strip()
flows['OA of workplace code'] = flows['OA of workplace code'].astype(str).str.strip()
lookup_lsoa['oa21cd'] = lookup_lsoa['oa21cd'].astype(str).str.strip()
lookup_lsoa['lsoa21cd'] = lookup_lsoa['lsoa21cd'].astype(str).str.strip()

In [5]:
#creat a mapping dictionary from OA to LSOA and lad namde
# create two simple dictionaries
oa_to_lsoa = lookup_lsoa.set_index('oa21cd')['lsoa21cd'].to_dict()
oa_to_lad  = lookup_lsoa.set_index('oa21cd')['ladnm'].to_dict()

# map origin
flows['origin_lsoa'] = flows['Output Areas code'].map(oa_to_lsoa)
flows['origin_lad']  = flows['Output Areas code'].map(oa_to_lad)

# map workplace
flows['workplace_lsoa'] = flows['OA of workplace code'].map(oa_to_lsoa)
flows['workplace_lad']  = flows['OA of workplace code'].map(oa_to_lad)


In [6]:
# quick check: how many matched / unmatched 
print("origin_lsoa matched:", flows['origin_lsoa'].notna().sum(), 
      "unmatched:", flows['origin_lsoa'].isna().sum())
print("Workplace_LSOA matched:", flows['workplace_lsoa'].notna().sum(), 
      "unmatched:", flows['workplace_lsoa'].isna().sum())

print("Workplace_LAD matched:", flows['workplace_lad'].notna().sum(), 
      "unmatched:", flows['workplace_lad'].isna().sum())
print("Origin_LSOA matched:", flows['origin_lsoa'].notna().sum(), 
      "unmatched:", flows['origin_lsoa'].isna().sum())


# filter rows where both Origin_LSOA and Workplace_LSOA and LADexist (no NaN)
flows_lsoa_complete = flows.dropna(subset=['origin_lsoa','workplace_lsoa','origin_lad','workplace_lad']).copy()
print("Rows with both LSOAs present:", len(flows_lsoa_complete))

origin_lsoa matched: 10740405 unmatched: 780
Workplace_LSOA matched: 10480006 unmatched: 261179
Workplace_LAD matched: 10480006 unmatched: 261179
Origin_LSOA matched: 10740405 unmatched: 780
Rows with both LSOAs present: 10479277


### extracting travel to work data for all 33 LDN bororughs

In [7]:
london_list = pd.read_excel('../data/london_borough.xlsx')
london_list.head()

,ladcode,ladnm
0,E09000001,City of London
1,E09000002,Barking and Dagenham
2,E09000003,Barnet
3,E09000004,Bexley
4,E09000005,Brent


In [8]:
london_boroughs = london_list['ladnm'].tolist()
london_boroughs

['City of London',
 'Barking and Dagenham',
 'Barnet',
 'Bexley',
 'Brent',
 'Bromley',
 'Camden',
 'Croydon',
 'Ealing',
 'Enfield',
 'Greenwich',
 'Hackney',
 'Hammersmith and Fulham',
 'Haringey',
 'Harrow',
 'Havering',
 'Hillingdon',
 'Hounslow',
 'Islington',
 'Kensington and Chelsea',
 'Kingston upon Thames',
 'Lambeth',
 'Lewisham',
 'Merton',
 'Newham',
 'Redbridge',
 'Richmond upon Thames',
 'Southwark',
 'Sutton',
 'Tower Hamlets',
 'Waltham Forest',
 'Wandsworth',
 'Westminster']

In [9]:
london_flows = flows_lsoa_complete[
    (flows_lsoa_complete['origin_lad'].isin(london_boroughs)) &
    (flows_lsoa_complete['workplace_lad'].isin(london_boroughs))
].copy()
print("Number of London flows extracted:", len(london_flows))


london_lsoa = pd.unique(
    london_flows[['origin_lsoa', 'workplace_lsoa']].values.ravel()
)
print("Number of London LSOAs:", len(london_lsoa))



Number of London flows extracted: 1360240
Number of London LSOAs: 4994


In [10]:
lookup_lsoa[lookup_lsoa['ladnm'].isin(london_boroughs)]['lsoa21cd'].nunique()

lookup_lsoa[
    lookup_lsoa['ladnm'].isin(london_boroughs)
].groupby('ladnm')['lsoa21cd'].nunique().sort_values(ascending=False)


ladnm
Croydon                   229
Barnet                    220
Bromley                   199
Ealing                    199
Wandsworth                186
Newham                    185
Enfield                   183
Brent                     181
Lambeth                   181
Lewisham                  175
Southwark                 173
Hillingdon                170
Tower Hamlets             169
Redbridge                 164
Greenwich                 164
Havering                  153
Hounslow                  150
Hackney                   149
Bexley                    148
Haringey                  147
Waltham Forest            146
Harrow                    144
Camden                    130
Islington                 126
Merton                    126
Sutton                    123
Westminster               123
Richmond upon Thames      115
Hammersmith and Fulham    115
Barking and Dagenham      115
Kensington and Chelsea    101
Kingston upon Thames       99
City of London              6
Name

In [11]:
london_flows

,Output Areas code,Output Areas label,OA of workplace code,OA of workplace label,Place of work indicator (4 categories) code,Place of work indicator (4 categories) label,Count,origin_lsoa,origin_lad,workplace_lsoa,workplace_lad
1,E00000001,E00000001,E00000001,E00000001,1,"Mainly working at or from home, No fixed place",64,E01000001,City of London,E01000001,City of London
2,E00000001,E00000001,E00004731,E00004731,3,Working in the UK but not working at or from home,1,E01000001,City of London,E01000956,Camden
3,E00000001,E00000001,E00006038,E00006038,3,Working in the UK but not working at or from home,1,E01000001,City of London,E01001205,Ealing
4,E00000001,E00000001,E00013547,E00013547,3,Working in the UK but not working at or from home,1,E01000001,City of London,E01002724,Islington
5,E00000001,E00000001,E00013548,E00013548,3,Working in the UK but not working at or from home,1,E01000001,City of London,E01002724,Islington
...,...,...,...,...,...,...,...,...,...,...,...
10187907,E00190469,E00190469,E00177676,E00177676,3,Working in the UK but not working at or from home,1,E01035709,Camden,E01032582,Lambeth
10187909,E00190469,E00190469,E00190415,E00190415,3,Working in the UK but not working at or from home,1,E01035709,Camden,E01000972,Camden
10187910,E00190469,E00190469,E00190441,E00190441,3,Working in the UK but not working at or from home,1,E01035709,Camden,E01035708,Camden
10187911,E00190469,E00190469,E00190459,E00190459,3,Working in the UK but not working at or from home,2,E01035709,Camden,E01004735,Westminster


In [12]:
london_flows = london_flows[
    london_flows["Place of work indicator (4 categories) label"]
    != "Mainly working at or from home, No fixed place"
].copy()


In [13]:
len(london_flows) # dropped 26357 data points

1333883

In [14]:
london_total_od = (
    london_flows
    .groupby(["origin_lsoa", "workplace_lsoa"])["Count"]
    .sum()
    .reset_index()
)

london_total_od.rename(columns={"Count": "flow"}, inplace=True)

print("Number of OD pairs:", len(london_total_od))
london_total_od.head()

Number of OD pairs: 929157


,origin_lsoa,workplace_lsoa,flow
0,E01000001,E01000001,3
1,E01000001,E01000002,10
2,E01000001,E01000512,1
3,E01000001,E01000851,2
4,E01000001,E01000855,4


In [15]:
# Step 1 — Remove self-loops and log transform
london_od = london_total_od[london_total_od['origin_lsoa'] != london_total_od['workplace_lsoa']].copy()
london_od['flow_log'] = np.log1p(london_od['flow'])

print(f"After removing self-loops: {len(london_od)}")
print(london_od.head())

After removing self-loops: 925406
  origin_lsoa workplace_lsoa  flow  flow_log
1   E01000001      E01000002    10  2.397895
2   E01000001      E01000512     1  0.693147
3   E01000001      E01000851     2  1.098612
4   E01000001      E01000855     4  1.609438
5   E01000001      E01000861     1  0.693147


### building POI feature matrix

In [16]:
import geopandas as gpd
# Read POIs (gpkg) and LSOA boundaries 
poi= gpd.read_file("../data/poi_uk.gpkg")       

# Replacing CSV with conversion to GeoDataFrame (using BNG eastings/northings) ---
LSOA_df = pd.read_csv("../data/Lower_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFE_(V10)_and_RUC.csv")


In [17]:
# Basic mapping from main_category to broader group
category_to_group = {

    # FOOD & DRINK
    'restaurant': 'FOOD_DRINK',
    'pub': 'FOOD_DRINK',
    'bar': 'FOOD_DRINK',
    'cafe': 'FOOD_DRINK',
    'coffee_shop': 'FOOD_DRINK',
    'fast_food': 'FOOD_DRINK',
    'fish_and_chips_restaurant': 'FOOD_DRINK',
    'chilean_restaurant': 'FOOD_DRINK',

    # SHOPPING / RETAIL
    'shopping': 'RETAIL',
    'shopping_mall': 'RETAIL',
    'grocery_or_supermarket': 'RETAIL',
    'convenience_store': 'RETAIL',
    'department_store': 'RETAIL',

    # LEISURE / SPORT
    'active_life': 'LEISURE_SPORT',
    'gym': 'LEISURE_SPORT',
    'sports_club': 'LEISURE_SPORT',
    'stadium': 'LEISURE_SPORT',
    'cinema': 'LEISURE_SPORT',
    'museum': 'LEISURE_SPORT',
    'tourist_attraction': 'LEISURE_SPORT',

    # BEAUTY & PERSONAL
    'beauty_and_spa': 'BEAUTY_PERSONAL',
    'hair_salon': 'BEAUTY_PERSONAL',
    'nail_salon': 'BEAUTY_PERSONAL',
    'spa': 'BEAUTY_PERSONAL',

    # PROFESSIONAL / BUSINESS
    'professional_services': 'PROFESSIONAL',
    'lawyer': 'PROFESSIONAL',
    'accountant': 'PROFESSIONAL',
    'consulting_agency': 'PROFESSIONAL',

    # HEALTH
    'hospital': 'HEALTH',
    'doctor': 'HEALTH',
    'dentist': 'HEALTH',
    'pharmacy': 'HEALTH',
    'elder_care_planning': 'HEALTH',

    # EDUCATION
    'school': 'EDUCATION',
    'primary_school': 'EDUCATION',
    'secondary_school': 'EDUCATION',
    'college': 'EDUCATION',
    'university': 'EDUCATION',
    'parenting_classes': 'EDUCATION',

    # AUTO
    'tire_repair_shop': 'AUTO',
    'car_repair': 'AUTO',
    'car_dealer': 'AUTO',
    'petrol_station': 'AUTO',

    # NATURE / OUTDOOR
    'park': 'NATURE_OUTDOOR',
    'sand_dune': 'NATURE_OUTDOOR',
    'beach': 'NATURE_OUTDOOR',
    'forest': 'NATURE_OUTDOOR',

    # RELIGION
    'church_cathedral': 'RELIGION',
    'church': 'RELIGION',
    'mosque': 'RELIGION',
    'synagogue': 'RELIGION',
    'temple': 'RELIGION',

    # ACCOMMODATION
    'hotel': 'ACCOMMODATION',
    'motel': 'ACCOMMODATION',
    'campground': 'ACCOMMODATION',
    'guest_house': 'ACCOMMODATION',
    'holiday_rental_home': 'ACCOMMODATION',
}


In [18]:
# Map each POI to a broad group
poi['poi_group'] = poi['main_category'].map(category_to_group)

# Check how many got mapped
print(poi['poi_group'].value_counts(dropna=False).head(20))

# keep only mapped rows to start simple
poi_mapped = poi[~poi['poi_group'].isna()].copy()


poi_group
NaN                2003083
FOOD_DRINK          118348
PROFESSIONAL         87777
BEAUTY_PERSONAL      87072
RETAIL               61169
LEISURE_SPORT        48277
HEALTH               41098
ACCOMMODATION        31645
RELIGION             20421
AUTO                 10648
NATURE_OUTDOOR        7396
EDUCATION             4248
Name: count, dtype: int64


In [19]:
all_lsoas_london = pd.Index(pd.concat([london_od['origin_lsoa'],
                                        london_od['workplace_lsoa']]).unique())

print(f"Total unique LSOAs: {len(all_lsoas_london)}")

Total unique LSOAs: 4994


In [20]:
# Pivot POI data to get counts per LSOA per category
poi_counts = (
    poi_mapped[poi_mapped['LSOA21CD'].isin(all_lsoas_london)]
    .groupby(['LSOA21CD', 'poi_group'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print(poi_counts.head())
print(f"Shape: {poi_counts.shape}")
print(f"POI categories: {poi_counts.columns.tolist()}")

poi_group   LSOA21CD  ACCOMMODATION  AUTO  BEAUTY_PERSONAL  EDUCATION  \
0          E01000001              1     0                5          0   
1          E01000002              2     1                3          1   
2          E01000003              0     0                1          0   
3          E01000005             10     1                8          2   
4          E01000006              1     0                0          0   

poi_group  FOOD_DRINK  HEALTH  LEISURE_SPORT  NATURE_OUTDOOR  PROFESSIONAL  \
0                   7       5              5               1            23   
1                  16       4             11               0            47   
2                   3       1              2               0             8   
3                  28       6             15               0            48   
4                   0       0              0               0             1   

poi_group  RELIGION  RETAIL  
0                 3       6  
1                 0       6  
2 

In [21]:
# Build feature matrix aligned to node index 
feature_df = pd.DataFrame({'LSOA21CD': list(all_lsoas_london)})

# Left join — missing LSOAs get 0 for all POI categories
feature_df = feature_df.merge(poi_counts, on='LSOA21CD', how='left')
feature_df = feature_df.fillna(0)

# Sanity check
assert len(feature_df) == len(all_lsoas_london), "Row count mismatch!"
assert list(feature_df['LSOA21CD']) == list(all_lsoas_london), "Node order mismatch!"

print(f"Feature matrix shape: {feature_df.shape}")  
print(f"LSOAs with all-zero POIs: {(feature_df.iloc[:, 1:].sum(axis=1) == 0).sum()}")
print(feature_df.head())

Feature matrix shape: (4994, 12)
LSOAs with all-zero POIs: 117
    LSOA21CD  ACCOMMODATION  AUTO  BEAUTY_PERSONAL  EDUCATION  FOOD_DRINK  \
0  E01000001            1.0   0.0              5.0        0.0         7.0   
1  E01000002            2.0   1.0              3.0        1.0        16.0   
2  E01000003            0.0   0.0              1.0        0.0         3.0   
3  E01000005           10.0   1.0              8.0        2.0        28.0   
4  E01000006            1.0   0.0              0.0        0.0         0.0   

   HEALTH  LEISURE_SPORT  NATURE_OUTDOOR  PROFESSIONAL  RELIGION  RETAIL  
0     5.0            5.0             1.0          23.0       3.0     6.0  
1     4.0           11.0             0.0          47.0       0.0     6.0  
2     1.0            2.0             0.0           8.0       1.0     1.0  
3     6.0           15.0             0.0          48.0       1.0     8.0  
4     0.0            0.0             0.0           1.0       0.0     0.0  


In [22]:
# Save raw POI counts for gravity model (before normalisation)
poi_cols       = [c for c in feature_df.columns if c != 'LSOA21CD']
poi_raw_counts = feature_df[poi_cols].values.copy()

In [23]:
#  Load IMD data (LSOA level)
imd_data = pd.read_csv('../data/LSOA_IMD2025_OSGB1936_-7580747902701771840.csv')  

# Load LSOA population data
pop_data = pd.read_csv('../data/new_population_data.csv')


In [24]:
pop_data['Total'] = pop_data['Total'].astype(str).str.replace(',', '').astype(float)
pop_london = pop_data[pop_data['LSOA 2021 Code'].isin(all_lsoas_london)][['LSOA 2021 Code', 'Total']].copy()
pop_london = pop_london.rename(columns={'LSOA 2021 Code': 'LSOA21CD', 'Total': 'population'})

imd_london = imd_data[imd_data['LSOA Code'].isin(all_lsoas_london)][['LSOA Code', 'IMD Decile']].copy()
imd_london = imd_london.rename(columns={'LSOA Code': 'LSOA21CD', 'IMD Decile': 'imd_decile'})


In [25]:
pop_london

,LSOA21CD,population
28761,E01000001,1573.0
28762,E01000002,1407.0
28763,E01000003,1610.0
28764,E01000005,1104.0
28765,E01032739,1633.0
...,...,...
33750,E01035718,2659.0
33751,E01035719,1269.0
33752,E01035720,1229.0
33753,E01035721,2336.0


In [26]:
imd_london

,LSOA21CD,imd_decile
0,E01000001,8
1,E01000002,10
2,E01000003,8
3,E01000005,5
4,E01000006,4
...,...,...
33711,E01035718,8
33712,E01035719,8
33713,E01035720,6
33714,E01035721,4


In [27]:
# Align both to node order
eval_df = pd.DataFrame({'LSOA21CD': list(all_lsoas_london)})
eval_df = eval_df.merge(pop_london, on='LSOA21CD', how='left')
eval_df = eval_df.merge(imd_london, on='LSOA21CD', how='left')
eval_df = eval_df.fillna(0)

population = eval_df['population'].values
imd_decile = eval_df['imd_decile'].values.astype(int)

print(f"Population matched: {(population > 0).sum()} / {len(population)}")
print(f"IMD matched:        {(imd_decile > 0).sum()} / {len(imd_decile)}")
print(f"Population range:   {population.min():.0f} - {population.max():.0f}")
print(f"IMD decile range:   {imd_decile.min()} - {imd_decile.max()}")

Population matched: 4994 / 4994
IMD matched:        4994 / 4994
Population range:   974 - 4380
IMD decile range:   1 - 10


## GRAVITYMODEL

In [28]:
# Population
X_train, X_test, y_train, y_test = train_test_split(
    poi_raw_counts, population, test_size=0.2, random_state=42)
grav_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"\nGravity; Population R²: {grav_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    poi_raw_counts, imd_decile, test_size=0.2, random_state=42)
rf_grav       = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
grav_imd_pred = rf_grav.predict(X_test)
grav_imd_acc  = accuracy_score(y_test, grav_imd_pred)
grav_imd_f1   = f1_score(y_test, grav_imd_pred, average='weighted')
print(f"Gravity → IMD Accuracy: {grav_imd_acc:.4f} | F1: {grav_imd_f1:.4f}")


Gravity; Population R²: 0.0542
Gravity → IMD Accuracy: 0.1241 | F1: 0.1154


# ML

In [29]:
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

#### build node index and PyG graph

In [30]:
# Node index 
lsoa_to_idx = {lsoa: i for i, lsoa in enumerate(all_lsoas_london)}

london_od['src'] = london_od['origin_lsoa'].map(lsoa_to_idx)
london_od['dst'] = london_od['workplace_lsoa'].map(lsoa_to_idx)

# Normalise POI for DGI 
x = torch.tensor(feature_df[poi_cols].values, dtype=torch.float)
x = torch.log1p(x)
x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-8)

# PyG graph 
edge_index  = torch.tensor(london_od[['src', 'dst']].values.T, dtype=torch.long)
edge_weight = torch.tensor(london_od['flow_log'].values, dtype=torch.float)
data        = Data(x=x, edge_index=edge_index, edge_weight=edge_weight)

print(f"Nodes:         {data.num_nodes}")
print(f"Edges:         {data.num_edges}")
print(f"Node features: {data.x.shape}")
print(f"x range:       [{data.x.min():.3f}, {data.x.max():.3f}]")
print(f"Any NaN:       {torch.isnan(data.x).any()}")

Nodes:         4994
Edges:         925406
Node features: torch.Size([4994, 11])
x range:       [-1.113, 9.063]
Any NaN:       False


#### Train DGI

In [31]:


# Encoder 
class DGIEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.prelu = torch.nn.PReLU()

    def forward(self, x, edge_index, edge_weight=None):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        x = self.prelu(x)
        return x

def corruption(x, edge_index, edge_weight=None):
    return x[torch.randperm(x.size(0))], edge_index, edge_weight

encoder = DGIEncoder(in_channels=data.x.shape[1],
                     hidden_channels=128, out_channels=64)

dgi = DeepGraphInfomax(
    hidden_channels=64,
    encoder=encoder,
    summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
    corruption=corruption
)

optimizer = torch.optim.Adam(dgi.parameters(), lr=0.001)

# Training 
dgi.train()
for epoch in range(300):
    optimizer.zero_grad()
    pos_z, neg_z, summary = dgi(data.x, data.edge_index, data.edge_weight)
    loss = dgi.loss(pos_z, neg_z, summary)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

# Extract embeddings    
dgi.eval()
with torch.no_grad():
    embeddings = encoder(data.x, data.edge_index, data.edge_weight)

emb = embeddings.numpy()
print(f"\nEmbeddings: {emb.shape}")
print(f"Range:      [{emb.min():.3f}, {emb.max():.3f}]")
print(f"Std:        {emb.std():.4f}")

Epoch   0 | Loss: 1.3256
Epoch  50 | Loss: 0.0365
Epoch 100 | Loss: 0.0466
Epoch 150 | Loss: 0.0350
Epoch 200 | Loss: 0.0486
Epoch 250 | Loss: 0.0247

Embeddings: (4994, 64)
Range:      [-1.205, 8.546]
Std:        0.3981


####  DGI: Population & IMD prediction 

In [32]:

# Population
X_train, X_test, y_train, y_test = train_test_split(
    emb, population, test_size=0.2, random_state=42)
dgi_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"DGI → Population R²: {dgi_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    emb, imd_decile, test_size=0.2, random_state=42)
rf_dgi       = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
dgi_imd_pred = rf_dgi.predict(X_test)
dgi_imd_acc  = accuracy_score(y_test, dgi_imd_pred)
dgi_imd_f1   = f1_score(y_test, dgi_imd_pred, average='weighted')
print(f"DGI     → IMD Accuracy: {dgi_imd_acc:.4f} | F1: {dgi_imd_f1:.4f}")

DGI → Population R²: 0.0892
DGI     → IMD Accuracy: 0.1832 | F1: 0.1694


In [33]:
lsoa_centroids = pd.read_csv('../data/LSOA_PopCentroids_EW_2021_V4_-1224685956027462334.csv')

In [34]:
lsoa_centroids

,FID,LSOA21CD,GlobalID,GlobalID_2,x,y
0,1,E01000001,38ad8fe8-682a-4533-b51a-16c9ca366294,dd315ba1-4937-466d-8c6a-cf5d42615b0a,532181.7642,181792.2818
1,2,E01018945,0681c038-f7dd-4a2e-83c1-c8d3b32443f6,3ae37539-20db-4f78-9537-61bfddacfce5,202699.4208,65668.0221
2,3,E01000120,e5ffc70b-0e24-40a9-afe8-13da1ec4032e,78bebe76-7b81-4351-bc3b-0b3ac11ca8e5,528823.5718,194238.0466
3,4,E01031487,17c7f13f-1a6a-45c5-a5cc-2c50e725c799,e79a12bf-45a3-4649-b1fe-8c5a93dca04a,486282.4085,106971.9143
4,5,E01021013,f55e8508-e975-4025-896c-a270de41fc79,054afc3d-7ce7-42fe-acf2-00789f24448b,583256.5228,111412.9755
...,...,...,...,...,...,...
35667,35668,E01023639,acc5933e-f958-4d5e-9d7c-bf80d3b3a8e4,7e1d7541-259b-42fb-bc83-7329f00e5267,522759.6016,231115.0000
35668,35669,E01035505,8cd1a7c3-774b-445d-9586-0d3cd2ec8816,fc470564-4157-476f-b15e-627fd8e8a6e5,455489.7686,339157.6943
35669,35670,E01012119,3e2e1865-4904-4bae-be77-16dc09da2363,62dfbe5d-e5a2-4c65-8194-89d88cb70cda,460894.7087,516424.3090
35670,35671,E01026708,02dfb6da-68f7-4b30-a2ab-ca024fcf491e,0c0a9816-806a-44b5-9022-ba336e3d54ba,564232.3414,323205.5021


In [35]:
centroid_london = lsoa_centroids[lsoa_centroids['LSOA21CD'].isin(all_lsoas_london)].copy()
coord_df = pd.DataFrame({'LSOA21CD': list(all_lsoas_london)})
coord_df = coord_df.merge(centroid_london[['LSOA21CD', 'x', 'y']],
                           on='LSOA21CD', how='left')

print(f"Centroids matched: {coord_df['x'].notna().sum()} / {len(all_lsoas_london)}")

Centroids matched: 4994 / 4994


In [36]:
coord_df.head()

,LSOA21CD,x,y
0,E01000001,532181.7642,181792.2818
1,E01000002,532501.3815,181755.6118
2,E01000003,532203.9352,182044.7623
3,E01000005,533746.3486,181107.5513
4,E01000006,544911.7299,184310.7291


In [37]:
london_od.head()

,origin_lsoa,workplace_lsoa,flow,flow_log,src,dst
1,E01000001,E01000002,10,2.397895,0,1
2,E01000001,E01000512,1,0.693147,0,492
3,E01000001,E01000851,2,1.098612,0,816
4,E01000001,E01000855,4,1.609438,0,818
5,E01000001,E01000861,1,0.693147,0,824


In [38]:
coords  = torch.tensor(coord_df[['x', 'y']].values, dtype=torch.float)
src_idx = torch.tensor(london_od['src'].values, dtype=torch.long)
dst_idx = torch.tensor(london_od['dst'].values, dtype=torch.long)

dist     = torch.sqrt(((coords[src_idx] - coords[dst_idx]) ** 2).sum(dim=1))
dist_log = torch.log1p(dist)
dist_log = (dist_log - dist_log.mean()) / (dist_log.std() + 1e-8)

od_features = torch.cat([
    embeddings[src_idx],
    embeddings[dst_idx],
    dist_log.unsqueeze(1)
], dim=1).numpy()

flow_target = london_od['flow_log'].values

X_train, X_test, y_train, y_test = train_test_split(
    od_features, flow_target, test_size=0.2, random_state=42)

flow_r2     = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))


print(f"DGI + Distance → Flow R²:  {flow_r2:.4f}")


DGI + Distance → Flow R²:  0.2561


In [39]:
# ── Gravity Model Flow Prediction ─────────────────────────────────────────────
# Features: log(pop_origin) + log(pop_destination) + log(distance)

# Get population for each OD pair
pop_series = pd.Series(population, index=list(all_lsoas_london))

origin_pop = pop_series[london_od['origin_lsoa']].values
dest_pop   = pop_series[london_od['workplace_lsoa']].values

# Log transform population
log_origin_pop = np.log1p(origin_pop)
log_dest_pop   = np.log1p(dest_pop)

# Distance already computed — use dist_log from earlier
gravity_features = np.column_stack([
    log_origin_pop,
    log_dest_pop,
    dist_log.numpy()
])

print(f"Gravity features shape: {gravity_features.shape}")
print(f"Any NaN: {np.isnan(gravity_features).any()}")

# Train/test split and predict
flow_target = london_od['flow_log'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    gravity_features, flow_target, test_size=0.2, random_state=42)

grav_flow_r2 = r2_score(y_te,
    LinearRegression().fit(X_tr, y_tr).predict(X_te))

print(f"\nGravity Model → Flow R²: {grav_flow_r2:.4f}")
print(f"DGI + Distance → Flow R²:  {flow_r2:.4f}")

Gravity features shape: (925406, 3)
Any NaN: False

Gravity Model → Flow R²: 0.0609
DGI + Distance → Flow R²:  0.2561


In [40]:
results_all = pd.DataFrame([
    ("Population",      "Gravity Model (POI)", "R²",       grav_pop_r2),
    ("Population",      "DGI Emb + POI",      "R²",       dgi_pop_r2    ),
    ("IMD Decile",      "Gravity Model (POI)", "Accuracy", grav_imd_acc ),
    ("IMD Decile",      "DGI Emb + POI",      "Accuracy", dgi_imd_acc   ),
    ("Flow Prediction", "DGI (Embedding + Distance)",      "R²",     flow_r2      ),
    ("Flow Prediction", "Gravity Model (Population + Distance)",    "R²", grav_flow_r2 ), 

], columns=["Task", "Model", "Metric", "London"])

print(results_all.to_string(index=False))


           Task                                 Model   Metric   London
     Population                   Gravity Model (POI)       R² 0.054239
     Population                         DGI Emb + POI       R² 0.089248
     IMD Decile                   Gravity Model (POI) Accuracy 0.124124
     IMD Decile                         DGI Emb + POI Accuracy 0.183183
Flow Prediction            DGI (Embedding + Distance)       R² 0.256139
Flow Prediction Gravity Model (Population + Distance)       R² 0.060897
